In [1]:
import sys
import json
from pathlib import Path
sys.path.append(str(Path("../").resolve()))


import pandas as pd
import numpy as np


import matplotlib as plt
import seaborn as sb


from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold


from sklearn.linear_model import LogisticRegression


from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.metrics import fbeta_score
from sklearn.metrics import make_scorer


from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline


from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.feature_selection import RFE


from src.prerocessing import get_scalar, scale_df
from src.models import lr_model
from src.models import Wrapper_MLP
from src.models import sgd_model
from src.models import lasso_model


from src.feature_selection import *


1) Load JSON
2) Feature selection
3) Data Transform
4) Test-Train split
5) Training with specific model
6) Evaluate
7) Save evaluate vals.

## 1) Load JSON & df
- vsetky nastavenia su realizovane v conf.json, a teda ak chceme nieco menit v experimentalnej pipeline, treba zmenit `conf.json` a spustit ju

In [2]:
conf_path: str = "../../conf/conf.json"
df_path: str = "../../data/clean_ds.xlsx"

# --- dataframe ---
df = pd.read_excel(df_path)


# --- JSON ---
with open(conf_path, 'r') as file:
    data = json.load(file)

#print(json.dumps(data))

config: dict = data["config"][0]
n_netwrok: dict = data["n_network"][0]
linear: dict = data["linear"][0]

### 1.1) Rozdelenie datasetu na atributya predikovanu hodnotu

In [3]:
X = df.drop('referred_CXL',
            axis=1
            )
y = df.referred_CXL

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=config["test_size"],
    random_state=config["random_state"]
    )

## 2) Data Tramsformation

In [4]:
# --- scale ---
scaler = get_scalar(config["scaling"])
#X_train_scaled, X_test_scaled = scale_df(X_train, X_test, scalar)


## 3) Pipeline

### 3.1) Aplikovanie vybrateho modelu

In [5]:
# premenne
max_iter: int = config["epoch"]
random_state: int = config["random_state"]
model_type: str = config["model"]
n_features_to_selection: int = config["n_features_to_selection"]
n_splits: int = config["n_splits"]

input_dim: int = 57 #20 #58
output_dim: int = n_netwrok["output_dim"]
lr: float = n_netwrok["lr"]

# Linear model configuration
early_stopping: bool = bool(linear["early_stopping"])
learning_rate: str = linear["learning_rate"]
l1_ratio: float = linear["l1_ratio"]
class_weights = linear["class_weight"]
c: float = linear["c"]
penalty: str = linear["penalty"]
solver: str = linear["solver"]

#class_weights: dict = {0: 1.5,
#                       1: 1.0}


# preklad / vyber modelu
models_dict: dict = {
    "lr": lr_model(max_iter=max_iter,
                   random_state=random_state,
                   class_weight=class_weights,
                   C=c,
                   penalty=penalty,
                   solver=solver),
    "mlp": Wrapper_MLP(output_dim=output_dim,
                       lr=lr,
                       epochs=max_iter),
    "sgd": sgd_model(max_iter=max_iter,
                     random_state=random_state,
                     learning_rate=learning_rate,
                     early_stopping=early_stopping,
                     class_weight=class_weights,
                     l1_ratio=l1_ratio),
    "lasso": lasso_model(max_iter=max_iter,
                         random_state=random_state)
}

# aplikacia modelu
model = models_dict[model_type]

### 3.2) Aplikovanie pipeline

In [6]:
f2_score = make_scorer(fbeta_score, beta=2)

eval_metrics: dict = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "recall_weighted": "recall_weighted",
    "roc_auc": "roc_auc",
    "f2_score": f2_score
}


if model_type == "mlp":
    pipeline = Pipeline([
        ("scaler", scaler),
        ("classifier", model)
    ])
else:
    pipeline = Pipeline([
        ("scaler", scaler),
        ("feature_selection", RFE(
            estimator=model,
            n_features_to_select=n_features_to_selection

        )),
        ("classifier", model)
    ])

cv = StratifiedKFold(n_splits=n_splits,
                     shuffle=True,
                     random_state=random_state
                     )
results = cross_validate(pipeline,
                         X,
                         y,
                         cv=cv,
                         scoring=eval_metrics
                         )
print(results)

{'fit_time': array([0.2121501 , 0.14278412, 0.14456606, 0.13590384, 0.14353895,
       0.13866901, 0.13719821, 0.13652396, 0.13935614]), 'score_time': array([0.00671792, 0.0055449 , 0.00482988, 0.00511312, 0.00494218,
       0.00483108, 0.00480986, 0.00496793, 0.00481677]), 'test_accuracy': array([0.80434783, 0.7173913 , 0.79347826, 0.75      , 0.80434783,
       0.82417582, 0.8021978 , 0.79120879, 0.82417582]), 'test_precision': array([0.55      , 0.44444444, 0.55555556, 0.5       , 0.57575758,
       0.59375   , 0.57692308, 0.55555556, 0.59375   ]), 'test_recall': array([1.        , 0.72727273, 0.86956522, 0.73913043, 0.82608696,
       0.86363636, 0.68181818, 0.68181818, 0.86363636]), 'test_recall_weighted': array([0.80434783, 0.7173913 , 0.79347826, 0.75      , 0.80434783,
       0.82417582, 0.8021978 , 0.79120879, 0.82417582]), 'test_roc_auc': array([0.93441558, 0.8538961 , 0.89350977, 0.83175803, 0.910523  ,
       0.90052701, 0.83465086, 0.87088274, 0.90909091]), 'test_f2_score'

In [7]:
acc = results["test_accuracy"].mean() * 100
prec = results["test_precision"].mean() * 100
rec = results["test_recall"].mean() * 100
rec_weig = results["test_recall_weighted"].mean() * 100
roc_auc = results["test_roc_auc"].mean() * 100
f2_sc = results["test_f2_score"].mean() * 100

print(f"Accuracy: {acc:.2f}%")
print(f"Precision: {prec:.2f}%")
print(f"Recall: {rec:.2f}%")
print(f"Recall Weighted: {rec_weig:.2f}%")
print(f"ROC AUC: {roc_auc:.2f}%")
print(f"f2 score: {f2_sc:.2f}%")

Accuracy: 79.01%
Precision: 54.95%
Recall: 80.59%
Recall Weighted: 79.01%
ROC AUC: 88.21%
f2 score: 73.49%
